In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings("ignore")

In [8]:
data = pd.read_csv("patient_data_processed.csv")

X = data.drop(columns=['Patient', 'Stages'])
y = data['Stages']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [11]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=9))
    ]),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42
    ),
    
    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel='linear'))
    ]),
    
    "RBF SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel='rbf'))
    ])
}

In [12]:
results = []

for name, model in models.items():
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    cv_scores = cross_val_score(model, X, y, cv=5)
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    results.append([name, acc, prec, rec, f1, cv_mean, cv_std])

comparison_df = pd.DataFrame(results, columns=[
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "CV Mean Accuracy",
    "CV Std Dev"
])

comparison_df.sort_values(by="Accuracy", ascending=False)

,Model,Accuracy,Precision,Recall,F1-Score,CV Mean Accuracy,CV Std Dev
2,Random Forest,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
4,RBF SVM,1.000000,1.000000,1.000000,1.000000,0.982466,0.025560
1,KNN,0.994521,0.994679,0.994521,0.994540,0.920000,0.066872
0,Logistic Regression,0.972603,0.977326,0.972603,0.973299,0.911233,0.081598
3,Linear SVM,0.972603,0.977326,0.972603,0.973299,0.934247,0.065019


In [13]:
best_model_name = comparison_df.sort_values(by="Accuracy", ascending=False).iloc[0]["Model"]
best_model = models[best_model_name]

best_model.fit(X_train, y_train)
y_pred_best = best_model.predict(X_test)

print("Best Model:", best_model_name)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_best))

Best Model: Random Forest

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        67
           1       1.00      1.00      1.00       130
           2       1.00      1.00      1.00       120
           3       1.00      1.00      1.00        48

    accuracy                           1.00       365
   macro avg       1.00      1.00      1.00       365
weighted avg       1.00      1.00      1.00       365


Confusion Matrix:

[[ 67   0   0   0]
 [  0 130   0   0]
 [  0   0 120   0]
 [  0   0   0  48]]
